# dSABRE (BFSExt router): padded vs original circuits


The **original** circuits are in `data/in/circuits`. The **padded** circuits in `data/in/circuits_alt` declare extra idle qubits,
so the router has as many qubits to work with as BOLTSABRE.

**Result files.** Each one is a run of `experiments/run.py` with the BFSExt router (`dSABRE_BFSExt`) over all 30 circuits
in 9 suites (`qasm_25` to `qasm_1500`) on devices A–I, with 5 seeds per circuit.
All devices (A–I) are defined in `experiments/arch.py`.
The experiment code and results are available at https://github.com/sibishan/dsabre.

- `dsabre_results.json`: BFSExt router, original circuits (baseline, A)
- `dsabre_results_alt.json`: BFSExt router, padded circuits (compared, B)

**Metrics.** Each circuit uses its best seed (the lowest inter-core count, then the fewest SWAPs, out of 5 seeds), as `run.py` picks it.
- **EPR** = inter-core operations = teleports + telegates (`best.inter_core`)
- **SWAP** = intra-core SWAPs (`best.swaps`)
- **Δ (%)** = (B − A) / A × 100. A **Gmean** row covers one circuit group; the final row covers the whole suite (30 circuits).
  On a Gmean row, Δ is computed from the two geomeans, which is the same as the geomean of the per-circuit ratios.

In [15]:
!pip install pandas -qqq

In [16]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML

HERE = Path.cwd()
IN_DIR = HERE.parent / "in"
RESULTS = {
    # circuit set: file (both runs use the BFSExt router)
    "original": "dsabre_results.json",
    "padded":   "dsabre_results_alt.json",
}
GROUP_ORDER = ["qasm_25", "qasm_64", "qasm_80", "qasm_95", "qasm_200", "qasm_360",
               "qasm_500", "qasm_1000", "qasm_1500"]
DEVICE = {   # from experiments/arch.py
    "qasm_25":  "25 qubits, device A: four 4×4 cores, 2×2 core grid",
    "qasm_64":  "64 qubits, device B: four 5×5 cores, 2×2 core grid",
    "qasm_80":  "80 qubits, device C: four 6×5 cores, 2×2 core grid",
    "qasm_95":  "95 qubits, device D: six 5×5 cores, 3×2 core grid",
    "qasm_200": "200 qubits, device E: eight 7×5 cores, 4×2 core grid",
    "qasm_360": "360 qubits, device F: twelve 7×6 cores, 4×3 core grid",
    "qasm_500":  "500 qubits, device G: fifteen 8×6 cores, 5×3 core grid",
    "qasm_1000": "1000 qubits, device H: twenty 11×6 cores, 5×4 core grid",
    "qasm_1500": "1500 qubits, device I: thirty 11×6 cores, 6×5 core grid",
}
NAMES = {"ae": "AE", "ghz": "GHZ", "graphstate": "Graphstate", "qft": "QFT", "qnn": "QNN",
         "random": "Random", "qpeexact": "QPEexact", "su2random": "SU2Random"}

In [17]:
rows = []
for cset, fname in RESULTS.items():
    data = json.loads((HERE / fname).read_text())
    for batch in data["batches"]:
        for r in batch["results"]:
            best = r["best"]
            rows.append({
                    "set": cset,
                "group": batch["batch"],
                "circuit": r["file"].split("_nativegates")[0],
                "file": r["file"],
                "epr": best["inter_core"] if best else np.nan,
                "swaps": best["swaps"] if best else np.nan,
            })

df = pd.DataFrame(rows)
missing = df[df[["epr", "swaps"]].isna().any(axis=1)]
print(f"{len(df)} results loaded; {len(missing)} circuit(s) with no valid run")
if len(missing):
    display(missing)

60 results loaded; 0 circuit(s) with no valid run


## CX counts

**CX** is the number of `cx` lines in the circuit's `.qasm` file. `cx` is the only two-qubit gate in these circuits, and padding
adds only idle qubits, so the count is the same in the original and padded copy (asserted below).

In [18]:
def cx_count(path):
    return sum(1 for line in open(path) if line.startswith("cx "))


cx = {}
for g, c, f in df.drop_duplicates(["group", "circuit"])[["group", "circuit", "file"]].itertuples(index=False):
    n_orig = cx_count(IN_DIR / "circuits" / g / f)
    n_pad = cx_count(IN_DIR / "circuits_alt" / g / f)
    assert n_orig == n_pad, f"CX differs for {g}/{f}: {n_orig} vs {n_pad}"
    cx[(g, c)] = n_orig
print(f"CX counts identical in original and padded sets for all {len(cx)} circuits")

CX counts identical in original and padded sets for all 30 circuits


## Table builder

`compare(a, b)` pairs two runs circuit by circuit. `table_html` lays the result out like the paper's Table III. `geomean` needs
strictly positive values: a zero count raises an error instead of being silently dropped.

In [19]:
def geomean(s):
    s = pd.Series(s).dropna()
    if (s <= 0).any():
        raise ValueError(f"geomean needs positive values, got {s[s <= 0].to_dict()}")
    return float(np.exp(np.log(s).mean())) if len(s) else np.nan


def delta(b, a):
    return (b / a - 1) * 100


def run(cset):
    sub = df[df["set"] == cset]
    return sub.set_index(["group", "circuit"])[["epr", "swaps"]]


def compare(a, b):
    """One row per circuit present in both runs: columns epr_a, swaps_a, epr_b, swaps_b."""
    return run(a).join(run(b), lsuffix="_a", rsuffix="_b", how="inner").dropna()


def gmean_row(label, sub):
    ea, sa, eb, sb = (geomean(sub[c]) for c in ("epr_a", "swaps_a", "epr_b", "swaps_b"))
    return dict(name=label, cx=None, ea=ea, sa=sa, eb=eb, sb=sb, de=delta(eb, ea), ds=delta(sb, sa))


def table_rows(p):
    rows = []
    for g in GROUP_ORDER:
        if g not in p.index.get_level_values("group"):
            continue
        sub = p.xs(g, level="group")
        rows.append(("section", DEVICE[g]))
        for c, r in sub.iterrows():
            rows.append(("circuit", dict(
                name=NAMES.get(c, c), cx=cx[(g, c)], ea=r.epr_a, sa=r.swaps_a, eb=r.epr_b, sb=r.swaps_b,
                de=delta(r.epr_b, r.epr_a), ds=delta(r.swaps_b, r.swaps_a))))
        rows.append(("gmean", gmean_row(f"Gmean ({len(sub)})", sub)))
    rows.append(("suite", gmean_row(f"Gmean, suite ({len(p)})", p)))
    return rows


CSS = """<style>
.ptab-wrap{display:inline-block;background:#ffffff;padding:18px 24px;border-radius:4px;margin:8px 0 24px}
.ptab-wrap,.ptab-wrap *{color:#0b0b0b !important}
.ptab,.ptab tr,.ptab th,.ptab td,.ptab caption{background:#ffffff !important}
.ptab{font-family:'Times New Roman',Times,serif;font-size:13px;border-collapse:collapse;margin:0}
.ptab caption{caption-side:top;font-variant:small-caps;text-align:center;padding-bottom:8px;max-width:640px;line-height:1.35}
.ptab caption b{display:block;font-variant:normal;letter-spacing:.06em;font-weight:normal}
.ptab th,.ptab td{padding:2px 10px;text-align:right;white-space:nowrap}
.ptab th:first-child,.ptab td:first-child{text-align:left}
.ptab thead tr:first-child th{border-top:1.6px solid #0b0b0b}
.ptab thead tr:last-child th{border-bottom:1px solid #0b0b0b}
.ptab .grp th{font-weight:normal;font-variant:small-caps;border-bottom:.6px solid #0b0b0b;text-align:center}
.ptab .sec td{font-style:italic;text-align:left;padding-top:8px}
.ptab .gm td{border-top:.6px solid #0b0b0b;font-weight:600}
.ptab .gm td.d,.ptab .suite td.d{font-weight:800}
.ptab .suite td{border-top:1px solid #0b0b0b;border-bottom:1.6px solid #0b0b0b;font-weight:600}
</style>"""


def _n(v, gm):
    return "—" if v is None else (f"{v:,.1f}" if gm else f"{int(v):,}")


def _d(v):
    return f"{v:+.1f}".replace("-", "−")


def table_html(p, number, caption, a_label, b_label):
    head = (f"<thead><tr class='grp'><th></th><th></th><th colspan=2>{a_label}</th>"
            f"<th colspan=2>{b_label}</th><th></th><th></th></tr>"
            "<tr><th>Circuit</th><th>CX</th><th>EPR</th><th>SWAP</th><th>EPR</th><th>SWAP</th>"
            "<th>ΔEPR (%)</th><th>ΔSWAP (%)</th></tr></thead>")
    body = []
    for kind, r in table_rows(p):
        if kind == "section":
            body.append(f"<tr class='sec'><td colspan=8>{r}</td></tr>")
            continue
        gm = kind != "circuit"
        cls = {"circuit": "", "gmean": "gm", "suite": "suite"}[kind]
        body.append(
            f"<tr class='{cls}'><td>{r['name']}</td><td>{_n(r['cx'], False)}</td>"
            f"<td>{_n(r['ea'], gm)}</td><td>{_n(r['sa'], gm)}</td>"
            f"<td>{_n(r['eb'], gm)}</td><td>{_n(r['sb'], gm)}</td>"
            f"<td class='d'>{_d(r['de'])}</td><td class='d'>{_d(r['ds'])}</td></tr>")
    cap = f"<caption><b>TABLE {number}</b>{caption}</caption>"
    return HTML(CSS + f"<div class='ptab-wrap'><table class='ptab'>{cap}{head}<tbody>{''.join(body)}</tbody></table></div>")


COMMON = ("EPR is teledata plus telegate; SWAP counts intra-core SWAPs; each circuit uses its best of 5 seeds. "
          "Gmean Δ is computed from the two geomeans. ")
PADDING = ("Original circuits declare only their active qubits; padded circuits add idle qubits "
           "to match BoltSABRE's qubit budget. Positive Δ means padding costs more.")

## Table I: effect of padding, BFSExt router

`dsabre_results_alt` vs `dsabre_results`. Positive Δ means the padded circuits cost more.

In [20]:
table_html(compare("original", "padded"), "I",
           "Padded vs original circuits, BFSExt router. " + COMMON + PADDING, "Original", "Padded")

/var/folders/n1/sg7h_ykd3979zm0fkg2x8tx00000gn/T/ipykernel_81543/302562110.py:19: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  return run(a).join(run(b), lsuffix="_a", rsuffix="_b", how="inner").dropna()
